# Bluestock Mutual Fund Analytics — Day 1: Data Ingestion & Live NAV API

This notebook demonstrates dataset ingestion across all 10 raw CSV files, prints data shapes, column data types, initial records, data quality checks, AMFI code validation, and live NAV fetching via `mfapi.in` API.

In [1]:
import os
from pathlib import Path
import pandas as pd
import requests

RAW_DIR = Path('../data/raw')
raw_files = list(RAW_DIR.glob('*.csv'))
print(f'Total raw CSV datasets found: {len(raw_files)}')
for f in sorted(raw_files):
    print(f'  - {f.name}')

Total raw CSV datasets found: 10
  - 1788499980509-304c1255-08_investor_transactions.csv
  - 1788499982117-e3d6ab98-09_portfolio_holdings.csv
  - 1788499982615-f9647ab2-10_benchmark_indices.csv
  - 1788499983024-b042c300-01_fund_master.csv
  - 1788499983331-4389156d-02_nav_history.csv
  - 1788499984134-b0cbf625-03_aum_by_fund_house.csv
  - 1788499984405-d702a6c6-04_monthly_sip_inflows.csv
  - 1788499984721-4b860901-05_category_inflows.csv
  - 1788499985036-da4a0c4a-06_industry_folio_count.csv
  - 1788499985420-bb134abf-07_scheme_performance.csv


## Inspect Raw Datasets (Shape, Columns, Dtypes & Sample Records)

In [2]:
for file_path in sorted(raw_files):
    df = pd.read_csv(file_path)
    print('=' * 70)
    print(f'FILE: {file_path.name}')
    print(f'Rows: {len(df):,}, Columns: {len(df.columns)}')
    print('Columns:', list(df.columns))
    print('Data Types:')
    print(df.dtypes)
    print('Missing Values:', df.isna().sum().to_dict())
    print('Head:')
    print(df.head(2))
    print('\n')

FILE: 1788499980509-304c1255-08_investor_transactions.csv
Rows: 32,778, Columns: 13
Columns: ['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']
Data Types:
investor_id               str
transaction_date          str
amfi_code               int64
transaction_type          str
amount_inr              int64
state                     str
city                      str
city_tier                 str
age_group                 str
gender                    str
annual_income_lakh    float64
payment_mode              str
kyc_status                str
dtype: object
Missing Values: {'investor_id': 0, 'transaction_date': 0, 'amfi_code': 0, 'transaction_type': 0, 'amount_inr': 0, 'state': 0, 'city': 0, 'city_tier': 0, 'age_group': 0, 'gender': 0, 'annual_income_lakh': 0, 'payment_mode': 0, 'kyc_status': 0}
Head:
  investor_id transaction_date  amfi_code transaction_

FILE: 1788499983331-4389156d-02_nav_history.csv
Rows: 46,000, Columns: 3
Columns: ['amfi_code', 'date', 'nav']
Data Types:
amfi_code      int64
date             str
nav          float64
dtype: object
Missing Values: {'amfi_code': 0, 'date': 0, 'nav': 0}
Head:
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474


FILE: 1788499984134-b0cbf625-03_aum_by_fund_house.csv
Rows: 90, Columns: 5
Columns: ['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']
Data Types:
date                  str
fund_house            str
aum_lakh_crore    float64
aum_crore           int64
num_schemes         int64
dtype: object
Missing Values: {'date': 0, 'fund_house': 0, 'aum_lakh_crore': 0, 'aum_crore': 0, 'num_schemes': 0}
Head:
         date           fund_house  aum_lakh_crore  aum_crore  num_schemes
0  2022-03-31      SBI Mutual Fund            6.05     605000          186
1  2022-03-31  ICICI Prudential MF            4.65     465000          

## Fund Master AMFI Code Validation

In [3]:
fm_file = [f for f in raw_files if '01_fund_master' in f.name][0]
df_fm = pd.read_csv(fm_file)
print('Total Schemes in Master:', len(df_fm))
print('Unique AMFI Codes:', df_fm['amfi_code'].nunique())
print('Duplicate AMFI Codes:', df_fm['amfi_code'].duplicated().sum())
print('AMFI Code Range:', df_fm['amfi_code'].min(), 'to', df_fm['amfi_code'].max())

Total Schemes in Master: 40
Unique AMFI Codes: 40
Duplicate AMFI Codes: 0
AMFI Code Range: 100016 to 149324


## Live NAV API Integration Test (`mfapi.in`)

In [4]:
amfi_code = 125497  # HDFC Top 100
url = f'https://api.mfapi.in/mf/{amfi_code}'
try:
    res = requests.get(url, timeout=5)
    if res.status_code == 200:
        data = res.json()
        print('API Status:', data.get('status'))
        print('Meta Scheme Name:', data.get('meta', {}).get('scheme_name'))
        print('Latest NAV Record:', data.get('data', [])[0])
    else:
        print('API Status Code:', res.status_code)
except Exception as e:
    print('API Call Exception (Using Fallback):', e)

API Status: SUCCESS
Meta Scheme Name: SBI SMALL CAP FUND - Direct Plan - Growth
Latest NAV Record: {'date': '04-09-2026', 'nav': '216.65250'}
